# 10.4 优化 checklist 汇总（可打印）

五张清单，分别对应建模、Cube、Vector、Scalar、核间五个战场。每张清单都是**「问自己一个问题」**的形式——全书方法的最终压缩形态。

## 清单一：建模（动手前）

```
□ 实测画像拿到了吗？波动排查、多次取中位数了吗？
□ W（计算量）、Q（访存量）算对了吗？混合带宽用哪级？
□ I = W/Q 与拐点 I* = Pm/Bm 比过吗？属于哪条线？
□ 理论上限 P = min(Bm·I, Pm) 推出来了吗？差距量化了吗？
```

## 清单二：Cube

```
□ K 全载装得下吗？（能全载不切分）
□ 必须切时 kBase 取容量上限了吗？（L0A/L0B 方程 + 分形 16 对齐）
□ L0C 满了吗？满时先切 M/N 而不是硬切 K？
□ 尾块走掩码/降档/合并哪条路？会不会成长尾？
□ 选优计算在 Host Tiling 侧完成了吗？
```

## 清单三：Vector

```
□ VF 循环不变量外提了吗？（Duplicate/标量除法）
□ 尾块用 MaskReg 而不是拆两个循环？
□ 相邻 Load/Store 合并了吗？广播用专用 LoadDist 了吗？
□ 除法能变乘法吗？（x*(a/b) → 先算 a/b）
□ 数据干净（无 mask/pse/drop）且 S2 长 → 考虑 DN 了吗？
□ VF 建模做过吗？实测与建模偏差解释了吗？
□ 寄存器双缓冲编排了吗？
```

## 清单四：Scalar

```
□ 凡不依赖运行时数据的计算都上提 Host 了吗？
□ 循环内地址是增量加法而不是乘法吗？
□ 循环内分支掩码化/查表/模板特化了吗？
□ SetFlag/WaitFlag 有冗余对吗？能合并的等待合并了吗？
□ 循环次数 ≤ 8 时展开了吗？
```

## 清单五：核间（负载均衡）

```
□ 分核计量是有效基本块数（含 mask/section 修正）吗？
□ CoreWeight 与 per-core 实测对得上吗？
□ 行数 < 核数或权重方差大 → 合轴考虑了吗？
□ decode 并行度不足 → 切 S2 + merge 了吗？
□ merge 段数 = 核数吗？（切更细只涨 merge 成本）
□ gs1 解码用查找表上提了吗？跨 BN 核路径覆盖了吗？
```

## 动手实验：checklist 自检脚本

把清单做成交互式自检——用「是/否/不适用」回答，最后统计各战场的完成度：

In [ ]:
# 动手：交互式 checklist 自检（示例运行: 预填一份「典型新手术前」答卷）
checklists = {
    '建模': ['实测画像拿到', 'W/Q 算对', 'I 与拐点比较', '理论上限量化'],
    'Cube': ['K 全载评估', 'kBase 取容量上限', 'L0C 满先切 M/N', '尾块方案', '选优上提 Host'],
    'Vector': ['不变量外提', 'MaskReg 尾块', '搬运合并/广播', '除法变乘法', 'DN 评估', 'VF 建模', '双缓冲编排'],
    'Scalar': ['标量上提', '地址增量化', '分支消除', '同步精简', '短循环展开'],
    '核间': ['有效块计量', 'CoreWeight 对拍', '合轴评估', '切 S2+merge', '段数=核数', '解码查找表'],
}

# 模拟一份「还没系统学优化」的答卷: 1=是, 0=否, -1=不适用
answers = {
    '建模':   [1, 0, 0, 0],
    'Cube':   [0, 0, -1, -1, 0],
    'Vector': [0, 0, 0, 0, 0, -1, -1],
    'Scalar': [0, 0, 0, 0, 0],
    '核间':   [0, 0, 0, 0, -1, -1],
}

print(f"{'战场':<8}{'完成':>4}{'总项':>4}{'未做':>18}")
for area, items in checklists.items():
    ans = answers[area]
    done = sum(1 for a in ans if a == 1)
    total = sum(1 for a in ans if a != -1)
    todo = [it for it, a in zip(items, ans) if a == 0]
    print(f"{area:<8}{done:>4}{total:>4}   {('、'.join(todo))[:16]}")
print()
score = sum(1 for v in answers.values() for a in v if a == 1)
valid = sum(1 for v in answers.values() for a in v if a != -1)
print(f"总完成度: {score}/{valid} = {score/valid:.0%} —— 按未做项逐条回对应章节补课")


### 观察结论

五张清单可直接打印贴在工位。这份「术前答卷」里最先该补的是**建模清单**——它决定后面所有优化的方向。

## 本节要点

- 五张清单 = 五个战场（建模/Cube/Vector/Scalar/核间）的问题化压缩；
- 每个问题都能回溯到具体章节的具体方法；
- 用自检脚本把「学过」变成「做过」——完成度低于一半的战场优先补课。

## 小测验

1. 建模清单的四个问题分别出自哪一节？
2. Cube 清单里「L0C 满先切 M/N」的原因是什么？
3. Vector 清单里哪两项与 DN 相关的判断条件？
4. 核间清单里「段数 = 核数」这条为什么重要？

>>> 参考答案见 [answer/10.04_answer.txt](answer/10.04_answer.txt)
